# 01 - Exploratory Data Analysis and Feature Audit

Uses the **training split only** (so no information from validation/test shapes decisions). Run `python -m src.data` first.
Every figure is saved to `reports/figures/`. Interpretations are left as TBD for you to write after looking at your own output;
patterns are **associations, not causes**.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data
from src.config import FIGURES_DIR, NUMERIC_FEATURES, SENSITIVE_ATTRIBUTES

sns.set_theme(style="whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=150)
    print("saved", name)

train = data.load_split("train")
print(train.shape)
train.head()

## 1. Target distribution

In [ ]:
counts = train["Churn"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Stayed (0)", "Churned (1)"], counts.values, color=["#4C78A8", "#E45756"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v} ({v/len(train):.1%})", ha="center", va="bottom")
ax.set_title("Target distribution (training split)")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of customers")
save(fig, "fig_target_distribution.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

## 2. Missing values and data types

In [ ]:
audit = pd.DataFrame({"dtype": train.dtypes.astype(str), "n_missing": train.isna().sum(),
                      "n_unique": train.nunique()})
audit

In [ ]:
blank = train[train["TotalCharges"].isna()]
print("Rows with blank TotalCharges:", len(blank))
print("Their tenure values:", sorted(blank["tenure"].unique()))

Blank `TotalCharges` are kept as NaN and imputed later inside the pipeline using training data only. **Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

## 3. Numeric features by churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, NUMERIC_FEATURES):
    sns.histplot(data=train, x=col, hue="Churn", bins=30, stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(f"{col} by churn status")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
save(fig, "fig_charges_by_churn.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

In [ ]:
bins = pd.cut(train["tenure"], [-1, 6, 12, 24, 48, 72], labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
rate = train.groupby(bins, observed=True)["Churn"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(rate.index.astype(str), rate.values, color="#4C78A8")
ax.axhline(train["Churn"].mean(), color="black", linestyle="--", label="Overall churn rate")
ax.set_title("Churn rate by tenure band (months)")
ax.set_xlabel("Tenure band (months)")
ax.set_ylabel("Churn rate")
ax.legend()
save(fig, "fig_churn_by_tenure_bin.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=train, x="Churn", y="tenure", ax=ax)
ax.set_title("Tenure by churn status")
ax.set_xlabel("Churn (0 = stayed, 1 = churned)")
ax.set_ylabel("Tenure (months)")
save(fig, "fig_tenure_by_churn.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

**Caution:** in this snapshot, churners' tenure ends when they leave, so tenure may be partly outcome-influenced.

## 4. Churn rate by key categorical variables

In [ ]:
def churn_rate_plot(col, fname, figsize=(7, 4)):
    rate = train.groupby(col)["Churn"].agg(["mean", "size"]).sort_values("mean", ascending=False)
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(rate.index.astype(str), rate["mean"], color="#4C78A8")
    for i, (m, n) in enumerate(zip(rate["mean"], rate["size"])):
        ax.text(i, m, f"n={n}", ha="center", va="bottom", fontsize=8)
    ax.axhline(train["Churn"].mean(), color="black", linestyle="--", label="Overall")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Churn rate")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    ax.legend()
    save(fig, fname); plt.show()
    return rate

churn_rate_plot("Contract", "fig_churn_by_contract.png")

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

In [ ]:
churn_rate_plot("PaymentMethod", "fig_churn_by_paymentmethod.png", (8, 4))

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

In [ ]:
churn_rate_plot("InternetService", "fig_churn_by_internetservice.png")

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

In [ ]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), addons):
    rate = train.groupby(col)["Churn"].mean()
    ax.bar(rate.index.astype(str), rate.values, color="#4C78A8")
    ax.axhline(train["Churn"].mean(), color="black", linestyle="--")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Churn rate")
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
save(fig, "fig_churn_by_addons.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

Note: 'No internet service' customers are a different group; add-on comparisons are confounded by internet type and price.

## 5. Numeric correlations

In [ ]:
corr = train[NUMERIC_FEATURES + ["Churn"]].corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation of numeric features and churn")
ax.set_xlabel("Variable")
ax.set_ylabel("Variable")
save(fig, "fig_numeric_correlation.png"); plt.show()

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

`tenure` and `TotalCharges` are expected to be strongly correlated; keep this in mind when reading importances.

## 6. Sensitive attributes (fairness review)
Churn rates by group are shown only to understand the data, **not** to target customers by these attributes.

In [ ]:
rows = []
for col in SENSITIVE_ATTRIBUTES:
    g = train.groupby(col)["Churn"].agg(["mean", "size"]).reset_index()
    g.insert(0, "attribute", col); g = g.rename(columns={col: "group"})
    rows.append(g)
pd.concat(rows, ignore_index=True)

**Interpretation (fill in after running on the real data):** TBD. Describe the pattern as an *association*; do not claim a cause.

## 7. Feature availability and model inputs
See `reports/data_audit.md`. Model inputs: numeric `tenure`, `MonthlyCharges`, `TotalCharges`; the other 16 columns (except `customerID` and `Churn`) are categorical.
Write here which findings change your plan (if any): TBD.